In [ ]:
# === ARRANQUE EN COLAB: árbol de carpetas de la sesión =====================
# Este cuaderno se escribió para correr desde la carpeta `notebook/` de su
# sesión, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese árbol, así que aquí se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E05_churn_fraude"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesion EPE E5 - Predecir categorias: churn y fraude

**Curso "Herramientas de Ciencias de Datos" - Modalidad EPE - UPC - Facultad de Negocios**

Esta sesion cubre dos frentes de clasificacion: **regresion logistica** para el **churn** y
**clasificadores con clases desbalanceadas** para el **fraude**. Enfoque EPE: **intuicion y decision
de negocio** sobre el formalismo.

**Objetivos.** Construir modelos que predicen categorias (fuga de clientes, fraude) con regresion
logistica; interpretar en **probabilidad y odds** (odds ratio como factor de riesgo); manejar clases
**desbalanceadas** (por que la exactitud engana; SMOTE como idea); y evaluar con **matriz de
confusion, ROC-AUC, PR-AUC, precision y recall**, eligiendo el **umbral por costo** (FN vs FP).

**Mapa de la sesion.**
1. Churn con logistica: la probabilidad estimada y los odds (Telco).
2. Odds ratio como factor de retencion; matriz de confusion, ROC/AUC.
3. El umbral de decision por costo (un FN pesa mucho mas que un FP).
4. Fraude y desbalance 99:1: la paradoja de la accuracy; precision/recall; ROC-AUC vs PR-AUC.
5. SMOTE como idea: el trade-off recall / precision.
6. Cierre: recomendacion de negocio.

> **Datos:** Telco Customer Churn y Credit Card Fraud ULB. Las cifras de
> **churn se recalculan en vivo**; las de **fraude 99:1 usan resultados ya validados**
> (el completo ~150,8 MB nunca vive en OneDrive). Convencion del
> curso: los resultados van a `resultados/E05_resultados.xlsx` y las figuras se generan **leyendo
> ese Excel**.
>
> **Abrir en Colab:** https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E05_churn_fraude/notebook/EPE_S5_clasificacion.ipynb
>
> Material hermano: `laboratorio/GUIA_LABORATORIO_E05.docx`; plantillas `plantillas/evaluacion_clasificador.docx`
> y `plantillas/guia_odds_ratio.docx`; ejercicios `evaluacion/drills.docx`; entregable `evaluacion/entregable.docx`;
> fuentes las fuentes de actualidad de la sesión.

In [ ]:
# SKIP-LOCAL: solo Colab. En el venv del curso estas librerias ya estan instaladas.
# imbalanced-learn (SMOTE) NO se instala: en E05 el SMOTE NO se ejecuta; sus cifras 99:1
# ya estan validadas (bloque d). Solo hacen falta sklearn, statsmodels, pandas, openpyxl, matplotlib.
import sys, subprocess
if "google.colab" in sys.modules:
    PINS = ["scikit-learn==1.6.1", "statsmodels==0.14.6",   # pins reales del venv del curso
            "pandas>=2.0", "openpyxl>=3.1", "matplotlib>=3.7"]
    def _pip(paquetes):
        return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes]).returncode
    if _pip(PINS) != 0:  # reintento SIN fijar version (por si Colab ya trae otras versiones)
        print("Aviso: fallo la instalacion con versiones fijas; se reintenta sin fijar version.")
        _pip([q.split("==")[0].split(">=")[0] for q in PINS])

In [ ]:
# Librerías de la sesión (ya en el venv del curso)
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import statsmodels.formula.api as smf
from openpyxl import Workbook
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
UPC_ROJO, UPC_TINTA, UPC_GRIS = "#C8102E", "#1F2A44", "#8A8D8F"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG a 150 dpi y la muestra (backend Agg)."
    try: fig.tight_layout()
    except Exception: pass
    fig.savefig(ruta, dpi=150, bbox_inches="tight"); plt.close(fig); display(Image(str(ruta)))

print("Librerías cargadas. Listo para logística (churn) y clasificación con desbalance (fraude).")

In [ ]:
# Rutas robustas (funcionan con nbconvert local y en Colab) y carga de datos
def localizar_sesion():
    aqui = Path.cwd()
    for base in [aqui, *aqui.parents]:
        if base.name == "E05_churn_fraude" and (base / "la ficha de la sesiónmd").exists():
            return base
        cand = base / "Sesiones_EPE" / "E05_churn_fraude"
        if cand.exists(): return cand
        if (base / "la ficha de la sesiónmd").exists(): return base
    return None

SESION = localizar_sesion() or Path.cwd()
DATA = SESION / "data"
RESULTADOS = SESION / "resultados"; RESULTADOS.mkdir(exist_ok=True)
FIGURAS = SESION / "figuras"; FIGURAS.mkdir(exist_ok=True)
XLSX = RESULTADOS / "E05_resultados.xlsx"

URL_TELCO = ("https://raw.githubusercontent.com/treselle-systems/"
             "customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv")
def cargar_telco():
    p = DATA / "telco_churn.csv"
    return pd.read_csv(p) if p.exists() else pd.read_csv(URL_TELCO)

print("Sesion:", SESION)
print("Resultados ->", XLSX.name, "| Figuras ->", FIGURAS.name)

---
## a) Predecir una categoría: la regresión logística y la probabilidad

La regresión **lineal** (E2) predice un número; para predecir una **categoría** (se fuga / no se fuga,
fraude / legítima) se usa la **regresión logística**, que estima una **probabilidad** entre 0 y 1
mediante la curva **sigmoide**. De la probabilidad se pasa a los **odds** (razón de momios):
`odds = p / (1 - p)`. Un odds de 1 equivale a `p = 0,5`; un odds de 4 significa "4 veces más probable
que ocurra a que no" (`p = 0,8`). Los coeficientes de la logística se leen en el terreno de los odds.

> **💡 Intuición de negocio.** La sigmoide es el **traductor**: convierte un puntaje (cuánto pesa cada
> factor) en una **probabilidad accionable** entre 0 y 1. Con esa probabilidad se **ordena** la cartera
> por riesgo; el resto de la sesión es decidir **dónde cortar** (el umbral) según el costo del error.

> **Nota de alcance (EPE).** La logística es un **GLM** (familia binomial, enlace logit). Las variantes
> para **conteos** (Poisson) o **importes** (gamma), y los discriminantes **LDA/QDA**, quedan **fuera
> del alcance EPE**: solo se nombran. El foco es construir, interpretar y decidir.

**📖 De dónde viene esta herramienta, y por qué se lee en odds ratio.** La logística no nació para
clasificar: nació para tamizar factores de riesgo con un presupuesto limitado. Los datos que la
fijaron en la literatura proceden del estudio **CORIS** (Rossouw et al., 1983), titulado "Coronary
risk factor screening in three rural communities"; su continuación de 1993 declara la decisión que
dependía de esa línea de base: un programa de intervención comunitaria en el que la **visita cara a
cara** se reservaba a los individuos de alto riesgo, porque es cara y no alcanza para todos. La
pregunta real nunca fue "quién se infarta" sino **"cuánto multiplica el riesgo cada factor, y a quién
conviene priorizar"**: palabra por palabra, el problema de retención del bloque *c*. De ahí salen dos
decisiones de método. Se descarta la **recta sobre una respuesta 0/1**, porque devuelve valores fuera
de [0, 1] y porque la varianza de un Bernoulli, `p(1-p)`, nunca es constante. Y se elige el **logit**
frente al **probit**, que era el estándar de la época: las dos curvas casi coinciden, pero solo en el
logit `exp(b)` **es** un odds ratio, un multiplicador que se dice en voz alta ante quien firma el
presupuesto. La logística se impuso por ser **comunicable**, no por ajustar mejor.

**📖 Y por qué se la llama "un GLM".** **Nelder & Wedderburn (1972)** no traían datos: traían un
desorden. Hasta entonces el caso binario se llamaba *probit analysis*, el de conteos *contingency
tables* y el continuo *regresión*, cada uno con su algoritmo y su capítulo aparte. Su artículo mostró
que basta con declarar **tres piezas** (la familia de la respuesta, el predictor lineal y la función
de enlace) para ajustarlos todos con la misma máquina: la regresión lineal ponderada iterativa. Por
eso la logística **es** el GLM binomial con enlace logit, y las variantes para conteos (Poisson) o
importes (gamma) son ese mismo marco con otra familia. Desarrollo completo con las citas verificadas
en la ficha de la sesión de réplica del paper, "Sección 0".


In [ ]:
# Mini-demo: la sigmoide  sigma(z) = 1/(1+e^-z)  convierte un puntaje lineal en PROBABILIDAD (0,1)
def sigmoide(z): return 1 / (1 + np.exp(-z))
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.plot(z, sigmoide(z), color=UPC_ROJO, lw=2.2)
ax.axhline(0.5, ls="--", color=UPC_GRIS, lw=1); ax.axvline(0, ls=":", color=UPC_GRIS, lw=1)
# Punto calculado score -> probabilidad: z=1,05 (~ el beta de 'fibra óptica', cuyo OR = exp(1,05) = 2,85)
z0 = 1.05; p0 = float(sigmoide(z0))
ax.plot([z0], [p0], "o", color=UPC_TINTA, ms=8, zorder=5)
ax.annotate(f"sigma({z0}) = {p0:.3f}", (z0, p0), textcoords="offset points",
            xytext=(8, -16), color=UPC_TINTA, fontsize=10)
ax.set_xlabel("Puntaje lineal  z = b0 + b1*x1 + ...  (log-odds)")
ax.set_ylabel("Probabilidad estimada  p = sigma(z)")
ax.set_title("La logística predice una PROBABILIDAD (sigmoide)")
mostrar(fig, FIGURAS / "demo_sigmoide.png")
print(f"Sigmoide con un punto:  sigma(0) = {sigmoide(0.0):.3f}  (score 0 -> p = 0,5);  "
      f"sigma({z0}) = {p0:.3f}  (score {z0} -> p = {p0:.1%})")
print("De probabilidad a odds:  odds = p/(1-p).  p=0,5 -> odds=1 (par);  p=0,8 -> odds=4 (4 a 1).")
print("Nota: el churn Telco está DESBALANCEADO (26,6% de fuga ~ 2,76:1), NO equilibrado; por eso "
      "el umbral por costo importa y la exactitud sola no basta.")

**❓ Qué se quiere averiguar.** ¿Puede el modelo ordenar la cartera de mayor a menor riesgo de fuga, o su orden no es mejor que el de una moneda?

- **Qué decide:** la campaña de retención no alcanza para toda la cartera. Si el orden es bueno, la llamada va primero a quien efectivamente está a punto de irse; si no lo es, la empresa gasta lo mismo que si llamara al azar.
- **Antes de mirar el resultado:** el **AUC-ROC** contesta esto **sin fijar todavía ningún umbral**. Si resulta **~0,5**, el modelo ordena como una moneda y la lista priorizada carece de valor. Entre **0,7 y 0,8** el orden es aceptable; entre **0,8 y 0,9** es bueno, y ahí sitúa la literatura a este dataset (**~0,84-0,88**): quedar muy por debajo apuntaría a un problema del dato antes que del método. Un AUC alto todavía no dice **a quién llamar**, solo **en qué orden mirar**.

**🔎 Antes de modelar.** Se limpian los **11 `TotalCharges` en blanco** (clientes con `tenure = 0`) y se
codifican las categóricas en **one-hot**. Un dato sucio aquí se propaga a **todos** los odds ratio: la
calidad del dato prevalece sobre la del modelo.

In [ ]:
# a) Cargar y limpiar Telco (TotalCharges: 11 espacios en blanco de clientes con tenure=0 -> NaN)
tc = cargar_telco()
tc["TotalCharges"] = pd.to_numeric(tc["TotalCharges"], errors="coerce")
n_blanco = int(tc["TotalCharges"].isna().sum())
tc = tc.dropna(subset=["TotalCharges"]).drop(columns=["customerID"])
tc["Churn"] = (tc["Churn"] == "Yes").astype(int)
print(f"Telco: {tc.shape[0]} clientes | {n_blanco} 'TotalCharges' en blanco eliminados | tasa de fuga = {tc['Churn'].mean()*100:.1f}%")

In [ ]:
# a) Entrenar la logistica: one-hot + escalado + particion 80/20 estratificada (semilla 42)
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in tc.columns if c not in num + ["Churn"]]
X = pd.get_dummies(tc[num + cat], columns=cat, drop_first=True); y = tc["Churn"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
esc = StandardScaler().fit(Xtr[num]); Xtr = Xtr.copy(); Xte = Xte.copy()
Xtr[num] = esc.transform(Xtr[num]); Xte[num] = esc.transform(Xte[num])

clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
prob_te = clf.predict_proba(Xte)[:, 1]
auc_churn = float(roc_auc_score(yte, prob_te))
fpr, tpr, _ = roc_curve(yte, prob_te)
print(f"Entrenamiento: {len(ytr)} | prueba: {len(yte)}")
print(f"AUC-ROC (test) = {auc_churn:.4f}   [operativo; la literatura reporta ~0,84-0,88 en el mismo dataset]")

**📖 Los argumentos del modelo, en términos simples.** Tres ajustes del código que conviene entender:

- **`max_iter=1000`** (en `LogisticRegression`): tope de iteraciones del optimizador **lbfgs**. Con las variables escaladas la solución converge con holgura dentro de ese tope; subirlo solo evita un aviso si no convergiera.
- **`C=1.0`** (valor por defecto de `LogisticRegression`, no escrito de forma explícita): es el **inverso de la fuerza de regularización**, `C = 1/lambda`. Un `C` **menor** regulariza **más** (contrae los coeficientes hacia 0, útil con muchas variables o poca muestra); un `C` mayor regulariza menos. `C=1.0` aplica una regularización L2 moderada, la de referencia.
- **`disp=0`** (en `statsmodels ...fit(disp=0)`): **silencia** el registro de iteraciones del optimizador. No cambia el resultado; solo evita imprimir la traza de convergencia.
- **Notación `C(Contract)` / `C(InternetService)`** (fórmula de `statsmodels`/`patsy`): la **`C(...)`** marca la variable como **categórica**; `patsy` crea solo las indicadoras (one-hot) tomando una categoría como **referencia**. Por eso en la tabla de OR aparece `C(Contract)[T.Two year]`: es el efecto de "contrato a dos años" **frente a** la categoría base (mes a mes).

In [ ]:
# a) De la probabilidad a los ODDS: un cliente de ejemplo del conjunto de prueba
i = int(np.argmax(prob_te))
p_i = float(prob_te[i]); odds_i = p_i / (1 - p_i)
print(f"Cliente de mayor riesgo estimado en test:  p(fuga) = {p_i:.2f}  ->  odds = p/(1-p) = {odds_i:.1f} a 1")
print("La probabilidad vive en [0,1]; los odds, en [0, inf). Los coeficientes del modelo se interpretan en odds.")

**📖 Lectura de negocio (probabilidad y AUC).** El modelo entrega, para cada cliente, una **probabilidad
de fuga**. Con esa probabilidad se puede **ordenar** la cartera por riesgo. El **AUC = 0,836** resume
ese ordenamiento **sin depender del umbral**: es la probabilidad de que el modelo puntúe más alto a un
cliente que se fuga que a uno que se queda. Guía: 0,5 azar; 0,7-0,8 aceptable; **0,8-0,9 bueno**;
>0,9 excelente. Un AUC de 0,84 dice "ordena bien a los clientes por su riesgo de fuga"; **qué decisión
tomar** con ese orden es la matriz de confusión y el umbral (más abajo).

## b) Interpretar en odds ratio: los factores de retención

**💡 Intuición.** Un OR es un **multiplicador de riesgo**: dice por cuánto se **multiplican** las odds de
fuga cuando la variable sube una unidad, no cuántos puntos de probabilidad cambian.

Los coeficientes de la logística viven en **log-odds** y **no** se leen directamente: se **exponencian**.
El **odds ratio (OR) = exp(b)** es cuánto se **multiplican** los odds de fuga cuando la variable sube una
unidad. `OR > 1` = factor de **riesgo** (más fuga); `OR < 1` = factor **protector** (menos fuga); `OR = 1`
= sin efecto. Se ajusta un modelo **compacto** con predictores interpretables para leer sus OR con
**p-valor** e **intervalo de confianza** (si el IC del OR cruza 1, el efecto no es significativo).

**❓ Qué se quiere averiguar.** ¿Qué factor multiplica o divide el riesgo de fuga, y cuál de ellos puede modificar la empresa esta semana?

- **Qué decide:** de aquí se deriva la palanca de retención. Un factor que la empresa no controla solo informa; uno que sí controla (el tipo de contrato, el cargo mensual) se convierte en una oferta concreta.
- **Antes de mirar el resultado:** cada coeficiente se lee como **odds ratio**. Un **OR > 1** es factor de riesgo; un **OR < 1** es protector; y si su **intervalo de confianza cruza el 1**, el efecto no se distingue de cero y no sostiene ninguna oferta. La hipótesis de trabajo es que el **contrato largo** salga claramente protector: de ser así, la palanca es migrar al cliente de mes a mes hacia un contrato; si su OR quedara cercano a 1, habría que buscar la palanca en otra variable.

In [ ]:
# b) Odds ratio de los factores de fuga (modelo compacto e interpretable, statsmodels)
mlog = smf.logit("Churn ~ tenure + MonthlyCharges + C(Contract) + C(InternetService)", data=tc).fit(disp=0)
ci = mlog.conf_int()
or_tab = pd.DataFrame({
    "predictor": mlog.params.index,
    "coef": mlog.params.values.round(4),
    "odds_ratio": np.exp(mlog.params.values).round(3),
    "p_valor": mlog.pvalues.values.round(4),
    "or_ic_bajo": np.exp(ci.iloc[:, 0].values).round(3),
    "or_ic_alto": np.exp(ci.iloc[:, 1].values).round(3),
})
display(or_tab)
or_tenure_anio = float(np.exp(12 * mlog.params["tenure"]))
print(f"tenure: OR por mes = {np.exp(mlog.params['tenure']):.3f}; por AÑO = exp(12*coef) = {or_tenure_anio:.3f}"
      f"  -> cada año de antigüedad reduce ~{(1-or_tenure_anio)*100:.0f}% los odds de fuga")
print(f"pseudo-R2 de McFadden (modelo compacto) = {mlog.prsquared:.3f}  (en logística, 0,2-0,4 ya es buen ajuste)")

**📖 Recomendación de retención (leyendo los OR).** Los factores coinciden con la evidencia del sector
(las fuentes de actualidad de la sesión):

- **Contrato a dos años: OR ~ 0,18** (y a un año ~ 0,42) frente a *mes a mes*: **migrar a contratos
 largos** es la palanca de retención más fuerte (factor protector, p < 0,001).
- **`tenure` (antigüedad): OR ~ 0,97 por mes** = **~0,68 por año**: cuanto más antiguo el cliente,
 menor el riesgo. Los **primeros meses** son los críticos.
- **Internet de fibra óptica: OR ~ 2,8** frente a DSL: segmento de **alto riesgo** (precio/expectativa),
 foco de campañas.
- **`MonthlyCharges`:** su IC del OR **cruza 1** (p ~ 0,14) una vez que se controla por contrato y
 antigüedad: **no** es un factor por sí solo.

> **⚠️ Alerta (OR no es probabilidad, ni causa).** Un OR de 0,18 **no** es "-82 % de probabilidad":
> multiplica las **odds**, no la probabilidad (la sigmoide es no lineal). Y el OR es una **asociación**,
> no una **causa** (puede haber factores no medidos): para decidir una acción de retención se valida con
> un **experimento A/B** (E1); la estimación causal formal es de E6.

## c) Evaluar la decisión: matriz de confusión y el umbral por costo

Con la probabilidad hay que **decidir**: se fija un **umbral** `t` y se predice "se fuga" si `p >= t`.
Cada caso cae en la **matriz de confusión** (VP, VN, FP, FN). En churn, un **FN** (no detectar a quien
se va) suele costar mucho más que un **FP** (ofrecer retención a quien se quedaba): por eso el 0,5 por
defecto rara vez es el punto correcto.

> **⚠️ Alerta.** Dejar el umbral en **0,5 "porque es lo normal"** u optimizar la **exactitud** deja
> sin detectar FN de alto costo. El umbral se elige por **costo** (bloque siguiente) y se fija con **validación**,
> nunca se ajusta en función del resultado en **test** (eso es fuga de información).

**❓ Qué se quiere averiguar.** Con el umbral de 0,5 que el modelo trae por defecto, ¿a cuántos clientes que sí se van no se les llama?

- **Qué decide:** cada uno de esos casos es un **FN**: un cliente que se pierde sin que nadie lo intente. La cifra dice si el valor por defecto sirve para la campaña o si hay que moverlo.
- **Antes de mirar el resultado:** si el **recall** resultara cerca de **1**, el 0,5 ya captaría casi toda la fuga y no habría nada que ajustar. Si resulta **en torno a 0,6**, cuatro de cada diez clientes que se van quedan fuera de la lista de llamadas, y la decisión la habrá tomado un valor por defecto que nadie eligió.

In [ ]:
# c) Matriz de confusión al umbral por defecto (0,5)
cm05 = confusion_matrix(yte, (prob_te >= 0.5).astype(int))
tn, fp, fn, tp = cm05.ravel()
print("Matriz de confusión (umbral 0,5):")
print(f"                 Pred: se queda   Pred: se fuga")
print(f"Real: se queda        VN={tn:5d}        FP={fp:4d}")
print(f"Real: se fuga         FN={fn:5d}        VP={tp:4d}")
print(f"recall(sensibilidad) = {tp/(tp+fn):.3f} | precisión = {tp/(tp+fp):.3f} | especificidad = {tn/(tn+fp):.3f}")
print(f"Con el 0,5 por defecto se ESCAPAN {fn} clientes que sí se fugan (FN): recall bajo.")

**❓ Qué se quiere averiguar.** Si perder un cliente cuesta S/300 y llamarlo cuesta S/20, ¿a partir de qué probabilidad de fuga conviene contactar al cliente?

- **Qué decide:** es la regla de operación de la campaña, la única línea de todo el cuaderno que el área de retención aplica cada día.
- **Antes de mirar el resultado:** el óptimo teórico es `t* = C_FP/(C_FP+C_FN) = 20/320 ~ 0,06`, muy por debajo del 0,5. Conviene comprometerse antes de ver la tabla: si el barrido devolviera un óptimo **cercano a 0,5**, los dos errores costarían más o menos lo mismo y el valor por defecto valdría; como el FN cuesta **15 veces** más que el FP, el óptimo debe caer **muy abajo**, y bajar el umbral tiene que **subir el recall** y **bajar el costo total** aunque la exactitud empeore.

**🔎 El paso que decide.** Se **barre** el umbral y se elige el que **minimiza el costo esperado**
`C_FN*FN + C_FP*FP`, no el de mayor exactitud. Con un FN mucho más caro que un FP, el óptimo cae **muy
por debajo** de 0,5.

In [ ]:
# c) Elegir el UMBRAL por costo (drill 2): un FN cuesta mucho más que un FP
C_FN, C_FP = 300, 20   # S/300: margen del cliente perdido  vs  S/20: llamada de retención
filas = []
for t in np.arange(0.1, 0.91, 0.1):
    a, b, c, d = confusion_matrix(yte, (prob_te >= t).astype(int)).ravel()
    prec = d / (d + b) if (d + b) else 0.0
    rec = d / (d + c) if (d + c) else 0.0
    filas.append([round(float(t), 2), d, b, c, a, round(prec, 3), round(rec, 3), int(C_FN * c + C_FP * b)])
tabla_umbral = pd.DataFrame(filas, columns=["umbral", "VP", "FP", "FN", "VN", "precision", "recall", "costo"])
display(tabla_umbral)
t_opt = float(tabla_umbral.loc[tabla_umbral["costo"].idxmin(), "umbral"])
t_teo = C_FP / (C_FP + C_FN)
print(f"Umbral teórico  t* = C_FP/(C_FP+C_FN) = {t_teo:.3f}")
print(f"Umbral que MINIMIZA el costo en el barrido = {t_opt}  (muy por debajo del 0,5 por defecto)")

**💡 Lectura de negocio (el núcleo de la decisión).** Como un cliente perdido (S/300) cuesta ~15x una
llamada de retención (S/20), el umbral óptimo `t* = 20 / 320 ~ 0,06` cae muy por debajo de 0,5. En el
barrido, **bajar el umbral a ~0,10** eleva el **recall de 0,58 a ~0,95** (se capta casi toda la fuga) al
precio de más falsos positivos, y **minimiza el costo esperado** (de S/50 000 a S/16 720). La decisión no
maximiza la exactitud, sino que **minimiza el costo del error**; en la práctica se acota además por la
**capacidad operativa** de la campaña de retención.

---
## d) Fraude y clases desbalanceadas: por que la exactitud engana

En **deteccion de fraude** la clase de interes es **muy poco frecuente y costosa**. En el dataset ULB hay **492
fraudes en 284 807 transacciones = 0,172 %** (desbalance ~99,83 : 0,17). Un modelo que clasifique todo como
**"legitimo"** acierta el **99,83 %** de las etiquetas **sin detectar un solo fraude** (recall = 0):
es la **paradoja de la accuracy**. Por eso, en desbalance, la exactitud **no** se usa; se miran
**precision, recall y PR-AUC** de la clase rara.

> **⚠️ Alerta (la paradoja de la exactitud).** En 99:1, "acertar el 99,83 %" es el **peor** consejo: se
> logra sin detectar un solo fraude. En desbalance la exactitud **no** se reporta; mandan **precision,
> recall y PR-AUC** de la clase rara.

> **Cifras reales, no recalculadas.** El desbalance 99:1 y todas las metricas de fraude que siguen
> provienen de **resultados ya validados** sobre el **dataset completo** (284 807 x 31). La **muestra
> local** (26,7 % de fraude) sirve solo para **explorar los datos**: sus metricas no se usan como
> resultado (daria otro numero). El completo (~150,8 MB) **nunca** vive en OneDrive.

In [ ]:
# d) Un vistazo a los datos de fraude (muestra de laboratorio; NO es el 99:1 real)
mp = DATA / "creditcard_muestra.csv"
if not mp.exists():
    # Carpeta de datos ya distribuida en Drive junto al cuaderno: se intenta antes
    # de rendirse (evita dejar el ejercicio en blanco en Colab).
    try:
        cc = pd.read_csv("https://drive.usercontent.google.com/download"
                         "?id=1HZmHWoz5J_94qat66TGyqjePEWlC8-1d&export=download&confirm=t")
        cc.to_csv(mp, index=False, encoding="utf-8")
    except Exception as e:
        cc = None
        print(f"  ! Muestra no disponible (Drive falló: {e!r}); ejecutar data/descargar_datos.py.")
else:
    cc = pd.read_csv(mp)
if cc is not None:
    print(f"Muestra de laboratorio: {cc.shape[0]} tx x {cc.shape[1]} cols "
          f"| fraude en la MUESTRA = {cc['Class'].mean()*100:.1f}% (submuestreo para mirar; el real es 0,17%)")
    display(cc[["Time", "Amount", "Class"]].head())

**🔎 Resultados ya validados, no recalculo.** El 99:1 real **no** se recalcula sobre la muestra local
(submuestreada al 26,7 % solo para explorar los datos): se usan los **conteos ya validados** sobre el
dataset completo (284 807 x 31), que nunca vive en OneDrive. Usar el resultado ya validado es mas
honesto que recalcular sobre una muestra sesgada.

In [ ]:
# d) Cargar los resultados 99:1 ya validados (dataset completo). Nada se recalcula sobre la muestra.
FR = dict(acc_trivial=0.9983, recall_trivial=0.0, roc_auc_base=0.9567, pr_auc_base=0.708,
          recall_base=0.6149, precision_base=0.8505, recall_smote=0.8784, precision_smote=0.0645,
          n_total=284807, n_fraude=492)
cm_base = [[85279, 16], [57, 91]]           # [[VN, FP], [FN, VP]]
cm_smote = [[83410, 1885], [18, 130]]
bench = [["LogReg (base)", 0.8505, 0.6149, 0.7137, 0.708, 0.9567],
         ["GaussianNB", 0.0604, 0.8041, 0.1124, 0.0809, 0.9552],
         ["LogReg + SMOTE", 0.0645, 0.8784, 0.1202, 0.7062, 0.9672]]

print("Fuente de las cifras de fraude: resultados ya validados")
print(f"Fraude real: {FR['n_fraude']} en {FR['n_total']} tx = {FR['n_fraude']/FR['n_total']*100:.4f}%  (desbalance ~99,83:0,17)")
print(f"Trivial 'todo legitimo':  accuracy = {FR['acc_trivial']:.4f}   recall(fraude) = {FR['recall_trivial']:.4f}   <- la accuracy ENGANA")
print(f"Base (umbral 0,5):  precision = {FR['precision_base']:.4f}  recall = {FR['recall_base']:.4f}  ROC-AUC = {FR['roc_auc_base']:.4f}  PR-AUC = {FR['pr_auc_base']:.4f}")

## e) Evaluar en desbalance: ROC-AUC vs PR-AUC, y otros clasificadores

Con 0,17 % de fraude, la **tasa de falsos positivos** se diluye entre ~284 000 legítimas, así que el
**ROC-AUC se ve optimista (~0,96)** aunque el modelo dispare muchas falsas alarmas. El **PR-AUC**
(precision-recall) **sí penaliza** esas falsas alarmas: es el **número honesto** en 99:1. Su línea base
no es 0,5 sino la **prevalencia** (~0,0017). El mismo modelo da **ROC-AUC 0,957** pero **PR-AUC 0,708**:
la brecha **es** el síntoma del desbalance.

Un clasificador sencillo como **Naive Bayes** (mención breve) suele detectar bastantes fraudes
(recall alto) pero con **precisión baja** (muchas falsas alarmas) y **PR-AUC bajo**: se compara por
PR-AUC, no por accuracy ni ROC-AUC.

> **⚠️ Alerta (ROC-AUC vs PR-AUC).** Un ROC-AUC de 0,96 puede convivir con **miles** de falsas alarmas:
> la FPR se diluye entre ~284 000 legítimas. Se decide por **PR-AUC**, cuya línea base es la
> **prevalencia** (~0,0017), no 0,5.

**❓ Qué se quiere averiguar.** De los tres clasificadores, ¿cuál se lleva a producción, y cambia la respuesta según la métrica con que se les evalúe?

- **Qué decide:** el modelo que quedará a filtrar transacciones. Elegirlo por la métrica equivocada no da un modelo un poco peor: da **otro** modelo.
- **Antes de mirar el resultado:** si el **ROC-AUC** y el **PR-AUC** ordenaran igual, la métrica sería indiferente y el debate sería innecesario. Se espera que **no** coincidan: **GaussianNB** debería lucir un ROC-AUC alto (~0,96, casi como la logística) y a la vez un **PR-AUC muy bajo**, porque su precisión cae ante las falsas alarmas. La brecha entre esas dos columnas **es** el síntoma del desbalance.

In [ ]:
# e) Comparacion de clasificadores en el fraude 99:1 (se ordena por PR-AUC, no por accuracy)
bench_df = pd.DataFrame(bench, columns=["modelo", "precision", "recall", "f1", "pr_auc", "roc_auc"])
display(bench_df.sort_values("pr_auc", ascending=False).reset_index(drop=True))
print("Se elige por PR-AUC (metrica honesta en desbalance): la logistica lidera; GaussianNB tiene ROC-AUC alto pero PR-AUC bajo.")

**📖 Leyendo la comparación.** Ordenada por **PR-AUC** (la métrica honesta), la **logística** lidera;
**GaussianNB** exhibe un ROC-AUC alto pero **PR-AUC bajo** (multiplica las falsas alarmas). En 99:1, elegir por
accuracy o por ROC-AUC llevaría al modelo equivocado.

## f) SMOTE como idea: el trade-off recall / precisión

**SMOTE** genera ejemplos sintéticos de la clase rara **solo en el conjunto de entrenamiento** (nunca
antes del split: eso sería **fuga de información** que infla las métricas). Hace al modelo **más
sensible**: detecta más fraudes (**recall sube 0,61 -> 0,88**) pero multiplica las falsas alarmas
(**precisión cae 0,85 -> 0,06**). No es una mejora sin costo: es un **trade-off**: conviene si el costo de
un fraude no detectado domina y el equipo puede absorber más falsas alarmas; si no, se prefiere ajustar
el **umbral** o usar `class_weight`. La decisión es de **negocio**, no del modelo.

> **⚠️ Alerta (fuga por SMOTE antes de partir).** Aplicar SMOTE **antes** del split train/test contamina
> el test con ejemplos sintéticos e **infla** las métricas (un recall casi perfecto e irreal). SMOTE va
> **solo en train**; el test conserva la **prevalencia real**.

**❓ Qué se quiere averiguar.** SMOTE detecta más fraude; ¿a cuántas alertas obliga a cambio, y puede el equipo revisarlas?

- **Qué decide:** si se activa SMOTE o se deja el modelo base y se ajusta el umbral. Es una decisión de **capacidad operativa**, no de estadística: la toma quien paga las horas de los analistas que revisan cada alerta.
- **Antes de mirar el resultado:** el trade-off ya viene anunciado en las tasas (recall **0,61 -> 0,88**, precisión **0,85 -> 0,06**), pero una jornada de trabajo no se planifica con tasas sino con **conteos**. Si los **FN** bajan poco y los **FP** suben poco, SMOTE resulta casi sin costo. Si los FN bajan unas decenas mientras los **FP se multiplican por cien**, cada fraude adicional detectado costará muchas revisiones manuales, y la pregunta pasa a ser cuántas caben en el día del equipo.

In [ ]:
# f) Efecto de SMOTE: conteos reales de la matriz de confusion (base vs SMOTE, dataset completo)
print("Base (umbral 0,5):   VN={0:>6,}  FP={1:>5,} | FN={2:>3}  VP={3:>3}".format(cm_base[0][0], cm_base[0][1], cm_base[1][0], cm_base[1][1]))
print("Con SMOTE (0,5):     VN={0:>6,}  FP={1:>5,} | FN={2:>3}  VP={3:>3}".format(cm_smote[0][0], cm_smote[0][1], cm_smote[1][0], cm_smote[1][1]))
print(f"SMOTE atrapa mas fraudes (FN {cm_base[1][0]} -> {cm_smote[1][0]}) pero multiplica las falsas alarmas (FP {cm_base[0][1]} -> {cm_smote[0][1]:,}).")

**📖 El trade-off en conteos.** SMOTE baja los **FN de 57 a 18** (detecta más fraude) pero sube los
**FP de 16 a 1 885** (más alertas que revisar). Conviene solo si el costo de un fraude no detectado
domina y el equipo puede absorber esas alertas; si no, se ajusta el **umbral** o `class_weight`.

## g) Exportacion a Excel y figuras de resultados

Convencion del curso: los resultados y pruebas se vuelcan a `resultados/E05_resultados.xlsx`, y las
**figuras de resultados se generan LEYENDO ese Excel** (no desde objetos en memoria). El churn se
calculo en vivo; el fraude usa resultados ya validados (dataset completo).

In [ ]:
# Construir el Excel de resultados de la sesion (churn en vivo + fraude ya validado)
wb = Workbook()
ws = wb.active; ws.title = "churn_metricas"
ws.append(["metrica", "valor"])
ws.append(["auc_telco_churn", round(auc_churn, 4)])
ws.append(["n_test", int(len(yte))])
ws.append(["churn_rate", round(float(y.mean()), 4)])
ws.append(["pseudo_r2_mcfadden_compacto", round(float(mlog.prsquared), 4)])
ws.append(["or_tenure_por_anio", round(or_tenure_anio, 3)])

ws2 = wb.create_sheet("churn_odds_ratio")
ws2.append(["predictor", "coef", "odds_ratio", "p_valor", "or_ic_bajo", "or_ic_alto"])
for _, r in or_tab.iterrows():
    ws2.append([str(r["predictor"]), float(r["coef"]), float(r["odds_ratio"]),
                float(r["p_valor"]), float(r["or_ic_bajo"]), float(r["or_ic_alto"])])

ws3 = wb.create_sheet("churn_confusion_umbral")
ws3.append(["matriz_confusion_umbral_0.5", "pred_se_queda", "pred_se_fuga"])
ws3.append(["real_se_queda", int(tn), int(fp)])
ws3.append(["real_se_fuga", int(fn), int(tp)])
ws3.append([])
ws3.append(["umbral", "VP", "FP", "FN", "VN", "precision", "recall", "costo"])
for _, f in tabla_umbral.iterrows():
    ws3.append([float(f["umbral"]), int(f["VP"]), int(f["FP"]), int(f["FN"]), int(f["VN"]),
                float(f["precision"]), float(f["recall"]), int(f["costo"])])

ws4 = wb.create_sheet("churn_roc")
ws4.append(["fpr", "tpr"])
for a, b in zip(fpr, tpr):
    ws4.append([round(float(a), 5), round(float(b), 5)])

ws5 = wb.create_sheet("fraude_metricas")
ws5.append(["metrica", "valor"])
for k in ["acc_trivial", "recall_trivial", "precision_base", "recall_base",
          "roc_auc_base", "pr_auc_base", "precision_smote", "recall_smote"]:
    ws5.append([k, round(float(FR[k]), 4)])
ws5.append(["n_total", int(FR["n_total"])]); ws5.append(["n_fraude", int(FR["n_fraude"])])
ws5.append(["prevalencia_fraude", round(FR["n_fraude"] / FR["n_total"], 6)])

ws6 = wb.create_sheet("fraude_confusion")
ws6.append(["base_umbral_0.5", "pred_legitima", "pred_fraude"])
ws6.append(["real_legitima", int(cm_base[0][0]), int(cm_base[0][1])])
ws6.append(["real_fraude", int(cm_base[1][0]), int(cm_base[1][1])])
ws6.append([])
ws6.append(["post_SMOTE_umbral_0.5", "pred_legitima", "pred_fraude"])
ws6.append(["real_legitima", int(cm_smote[0][0]), int(cm_smote[0][1])])
ws6.append(["real_fraude", int(cm_smote[1][0]), int(cm_smote[1][1])])

ws7 = wb.create_sheet("fraude_benchmark")
ws7.append(["modelo", "precision", "recall", "f1", "pr_auc", "roc_auc"])
for row in bench:
    ws7.append(row)

wb.save(XLSX)
print("Excel guardado en:", XLSX)
print("Hojas:", wb.sheetnames)

### Figuras de resultados (leyendo el Excel)

In [ ]:
# Figura 1 - Curva ROC del churn (leída de churn_roc; AUC de churn_metricas)
roc = pd.read_excel(XLSX, sheet_name="churn_roc").dropna(subset=["fpr", "tpr"])
meta = pd.read_excel(XLSX, sheet_name="churn_metricas")
auc_l = float(meta.loc[meta["metrica"] == "auc_telco_churn", "valor"].iloc[0])
fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.plot(roc["fpr"], roc["tpr"], color=UPC_ROJO, lw=2.2, label=f"Logística (AUC = {auc_l:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color=UPC_GRIS, lw=1, label="Azar (AUC = 0,50)")
ax.set_xlabel("Tasa de falsos positivos (1 - especificidad)")
ax.set_ylabel("Sensibilidad (recall)")
ax.set_title("Curva ROC - modelo de fuga de clientes (test)")
ax.legend(loc="lower right", frameon=False)
mostrar(fig, FIGURAS / "churn_roc.png")

In [ ]:
# Figura 2 - Odds ratio de los factores de fuga con IC 95% (forest plot, leído de churn_odds_ratio)
orx = pd.read_excel(XLSX, sheet_name="churn_odds_ratio")
orx = orx[orx["predictor"] != "Intercept"].sort_values("odds_ratio")
sig = orx["p_valor"] < 0.05
col = [UPC_ROJO if s else UPC_GRIS for s in sig]
etq = [p.replace("C(", "").replace(")", "").replace("[T.", " = ").replace("]", "") for p in orx["predictor"]]
fig, ax = plt.subplots(figsize=(7.0, 4.0)); yy = np.arange(len(orx))
ax.errorbar(orx["odds_ratio"], yy,
            xerr=[orx["odds_ratio"] - orx["or_ic_bajo"], orx["or_ic_alto"] - orx["odds_ratio"]],
            fmt="none", ecolor=UPC_GRIS, elinewidth=1.5, capsize=3)
ax.scatter(orx["odds_ratio"], yy, color=col, s=55, zorder=3)
ax.axvline(1.0, ls="--", color=UPC_TINTA, lw=1)
ax.set_yticks(yy); ax.set_yticklabels(etq)
ax.set_xlabel("Odds ratio = exp(b)   (1 = sin efecto; <1 protege, >1 aumenta la fuga)")
ax.set_title("Factores de fuga de clientes (rojo = significativo, p < 0,05)")
mostrar(fig, FIGURAS / "churn_odds_ratio.png")

In [ ]:
# Figura 3 - Costo esperado vs umbral (leído de churn_confusion_umbral)
crudo = pd.read_excel(XLSX, sheet_name="churn_confusion_umbral", header=None)
ini = crudo.index[crudo[0] == "umbral"][0]
tab = crudo.iloc[ini + 1:].copy(); tab.columns = crudo.iloc[ini].values
tab = tab.dropna(subset=["umbral"]).astype({"umbral": float, "costo": float, "recall": float})
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.plot(tab["umbral"], tab["costo"], "-o", color=UPC_ROJO, lw=2, label="Costo esperado (S/)")
jmin = tab["costo"].idxmin()
ax.scatter([tab.loc[jmin, "umbral"]], [tab.loc[jmin, "costo"]], color=UPC_TINTA, s=120, zorder=5,
           label=f"Mínimo en umbral = {tab.loc[jmin, 'umbral']:.1f}")
ax.axvline(0.5, ls="--", color=UPC_GRIS, lw=1, label="Umbral por defecto (0,5)")
ax.set_xlabel("Umbral de decisión"); ax.set_ylabel("Costo esperado = C_FN*FN + C_FP*FP")
ax.set_title("El umbral que minimiza el costo está muy por debajo de 0,5")
ax.legend(frameon=False)
mostrar(fig, FIGURAS / "churn_costo_umbral.png")

In [ ]:
# Figura 4 - Matriz de confusión del churn como heatmap (leída del bloque de churn_confusion_umbral)
blk = pd.read_excel(XLSX, sheet_name="churn_confusion_umbral", header=None)
cm = np.array([[int(blk.iloc[1, 1]), int(blk.iloc[1, 2])], [int(blk.iloc[2, 1]), int(blk.iloc[2, 2])]])
fig, ax = plt.subplots(figsize=(4.6, 4.0)); ax.imshow(cm, cmap="Reds")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: se queda", "Pred: se fuga"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Real: se queda", "Real: se fuga"])
et = np.array([["VN", "FP"], ["FN", "VP"]])
for i in range(2):
    for j in range(2):
        c = "white" if cm[i, j] > cm.max() / 2 else UPC_TINTA
        ax.text(j, i, f"{et[i, j]}\n{cm[i, j]}", ha="center", va="center", color=c, fontsize=12, fontweight="bold")
ax.set_title("Matriz de confusión - churn (umbral 0,5)"); ax.grid(False)
mostrar(fig, FIGURAS / "churn_matriz_confusion.png")

In [ ]:
# Figura 5 - Fraude: matriz de confusión base vs SMOTE (leída de fraude_confusion)
cf = pd.read_excel(XLSX, sheet_name="fraude_confusion", header=None)
CMB = np.array([[int(cf.iloc[1, 1]), int(cf.iloc[1, 2])], [int(cf.iloc[2, 1]), int(cf.iloc[2, 2])]])
CMS = np.array([[int(cf.iloc[5, 1]), int(cf.iloc[5, 2])], [int(cf.iloc[6, 1]), int(cf.iloc[6, 2])]])
fig, axs = plt.subplots(1, 2, figsize=(9.4, 4.3))
for ax, cm, tit in zip(axs, [CMB, CMS], ["Base (umbral 0,5)", "Con SMOTE (umbral 0,5)"]):
    ax.imshow(cm, cmap="Reds")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: legítima", "Pred: fraude"], fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Real: legítima", "Real: fraude"], fontsize=9)
    et = np.array([["VN", "FP"], ["FN", "VP"]])
    for i in range(2):
        for j in range(2):
            c = "white" if cm[i, j] > cm.max() / 2 else UPC_TINTA
            ax.text(j, i, f"{et[i, j]}\n{cm[i, j]:,}", ha="center", va="center", color=c, fontsize=10, fontweight="bold")
    ax.set_title(tit); ax.grid(False)
fig.suptitle("Fraude: SMOTE detecta más fraudes (FN 57->18) pero dispara las falsas alarmas (FP 16->1 885)")
mostrar(fig, FIGURAS / "fraude_confusion_base_smote.png")

In [ ]:
# Figura 6 - SMOTE: precisión y recall de la clase fraude, base vs SMOTE (leído de fraude_metricas)
fm = pd.read_excel(XLSX, sheet_name="fraude_metricas"); mv = dict(zip(fm["metrica"], fm["valor"]))
base = [mv["precision_base"], mv["recall_base"]]; smote = [mv["precision_smote"], mv["recall_smote"]]
x = np.arange(2); w = 0.36
fig, ax = plt.subplots(figsize=(6.2, 4.0))
b1 = ax.bar(x - w/2, base, w, color=UPC_TINTA, label="Sin SMOTE (base)")
b2 = ax.bar(x + w/2, smote, w, color=UPC_ROJO, label="Con SMOTE")
for bars in (b1, b2):
    for r in bars:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.02, f"{r.get_height():.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(["Precisión", "Recall"]); ax.set_ylim(0, 1.08)
ax.set_ylabel("Valor sobre la clase fraude")
ax.set_title("SMOTE: el recall sube (0,61->0,88) y la precisión se desploma (0,85->0,06)")
ax.legend(frameon=False)
mostrar(fig, FIGURAS / "fraude_precision_recall_smote.png")

In [ ]:
# Figura 7 - Por qué el ROC-AUC engaña en 99:1: ROC-AUC vs PR-AUC (leído de fraude_metricas)
fig, ax = plt.subplots(figsize=(5.8, 4.0))
vals = [mv["roc_auc_base"], mv["pr_auc_base"]]
bars = ax.bar(["ROC-AUC", "PR-AUC"], vals, color=[UPC_GRIS, UPC_ROJO])
for r in bars:
    ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.02, f"{r.get_height():.3f}", ha="center", fontsize=10)
ax.axhline(mv["prevalencia_fraude"], ls="--", color=UPC_TINTA, lw=1,
           label=f"Línea base PR = prevalencia ({mv['prevalencia_fraude']:.4f})")
ax.set_ylim(0, 1.08); ax.set_ylabel("Área bajo la curva")
ax.set_title("En 99:1 el ROC-AUC engaña (0,96); el PR-AUC es el número honesto (0,71)")
ax.legend(frameon=False)
mostrar(fig, FIGURAS / "fraude_roc_vs_pr.png")

---
## 7. Drills (enunciados; se resuelven en `evaluacion/drills.docx`)

Se resuelven en parejas; el entregable es individual. Las soluciones son material docente.

1. **De coeficiente a probabilidad/odds.** Tomar el OR de un factor de fuga (p. ej. contrato a dos
   anios) de la tabla `churn_odds_ratio`, interpretarlo en una frase de negocio y decir si es
   significativo (IC del OR que no cruza 1). Convertir el OR de `tenure` a "por anio".
2. **Umbral por costo de falsos negativos.** Con el modelo de churn, suponer `C_FN = S/500` y
   `C_FP = S/25`. Calcular `t* = C_FP/(C_FP+C_FN)`, rehacer la tabla de umbral y elegir el que minimiza
   el costo. Comparar recall y numero de contactos frente al umbral 0,5.
3. **Con y sin SMOTE.** Con el caso de fraude (cifras ya validadas), comparar **precision y recall** de
   la clase fraude **antes y despues** de SMOTE; decidir si conviene, segun el costo de un fraude no
   detectado vs. el de una falsa alarma.

## 8. Cierre

### Entregable evaluable
Ajustar un **clasificador de churn** (Telco): interpretar los coeficientes en **odds ratio**, evaluar
con **matriz de confusión / ROC / AUC**, **elegir el umbral por costo** de los falsos negativos y
comunicar una **recomendación de retención**; y leer el caso **fraude** (desbalance: PR-AUC,
precisión/recall, efecto de SMOTE). Se entrega con la plantilla `plantillas/evaluacion_clasificador.docx`
y la guía `plantillas/guia_odds_ratio.docx`; enunciado y **rúbrica vigesimal (0-20)** en
`evaluacion/entregable.docx`. Pasos guiados en `laboratorio/GUIA_LABORATORIO_E05.docx`.

### Control corto
Habrá un **control corto** (8 preguntas) sobre logit/odds ratio, umbral por costo, la paradoja de la
accuracy, ROC-AUC vs PR-AUC y el efecto de SMOTE.

### Proyecto integrador
Esta sesión alimenta la fase de **Modelado**: un modelo de propensión (fuga / impago / fraude) con su
lectura en odds ratio y su umbral por costo es una pieza directa del proyecto transversal.

### Para seguir explorando (las fuentes de actualidad de la sesión)
- **Telcos e IA para predecir la fuga** - *RCR Wireless News*, 16/12/2025 (fuga 15-30 %; logística como línea base).
- **Retención en telecom con analítica predictiva** - *Frontiers in AI*, 29/08/2025 (mismo dataset IBM; logística **AUC 0,88**).
- **Fraude con tarjeta: pérdidas globales ~$33 mil millones** - *Nilson Report* via GlobeNewswire, 07/01/2026.
- **SMOTE y el trade-off recall/precisión en fraude** - *Frontiers in AI*, 08/10/2025 (logística: recall 100 % / precisión 24 %).
- **Data leakage por SMOTE antes del split** - *arXiv 2506.02703*, 03/06/2025 (99,9 % de recall engañoso).

> **Próximas sesiones EPE (solo se nombran).** E6 pronóstico y causalidad; E7 recomendación y proyecto integrador.